# PYTHIA CP5 generator-level jet check

Read the standalone PYTHIA CP5 generator analysis from CERNBox. The `local` source uses `$HOME/cernbox/ana/pPb8160/pythia/cp5tune/cp5_gen_dijet_histograms.root`. The workflow overlays inclusive-jet $\eta_{CM}$ shapes and Forward/Backward ratios in configurable jet-$p_T$ intervals, then plots the generator-dijet $\eta_{CM}^{dijet}$ distribution and Forward/Backward ratio in either compact test or standard $p_T^{ave}$ bins.

### Physics and statistical conventions

All projections use half-open momentum intervals $[p_{T,low},p_{T,high})$. Full $\eta$ distributions are normalized according to the user-facing `NORMALIZATION` setting; the default unit bin-content sum compares shapes. F/B is always constructed from the **unnormalized** stored forward and backward weighted yields. Since they are separate weighted populations, `TH1::Divide(..., '')` supplies standard independent-error propagation; binomial errors are neither used nor configurable. For these equal-energy pp collisions, $\eta_{CM}=\eta_{lab}$.

## Environment and imports

Start Jupyter from the repository root with `py-env/bin/python -m jupyter notebook`.

In [ ]:
# Cell role: initialize the project Python/PyROOT environment and shared helpers.
# Interpretation: no physics object is changed; ROOT ownership is configured before input is opened.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
from IPython.display import display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'CMakeLists.txt').is_file() and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate jetAnalysis. Start Jupyter from its root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS, STANDARD_DIJET_ETA_CUT_INDEX, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.data_distributions import forward_backward_from_full
from hist_analysis.python.dijet_closures import DijetClosureCurve, build_dijet_gen_comparisons
from hist_analysis.python.histogram_io import load_histogram
from hist_analysis.python.histogram_ops import normalize_histogram
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.projections import project_semantic_th2

ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration and input resolution

Set `INPUT_SOURCE` to `cernbox`, `local`, or `auto`. `CP5_FILE` overrides every default when the `CP5_GEN_ANALYSIS_FILE` environment variable is defined. The local input is `$HOME/cernbox/ana/pPb8160/pythia/cp5tune/cp5_gen_dijet_histograms.root`. Index 5 is the reference per-jet acceptance $|\eta_{CM}^{jet}|<1.9$.

In [ ]:
# Cell role: define user-facing inputs, binning, normalization, and plotting controls.
# Interpretation: these settings determine the selected CP5 sample and all displayed phase-space intervals.
INPUT_SOURCE = 'local'       # auto, cernbox, or local
INCLUSIVE_PT_BINS = ((50, 80), (120, 180), (250, 500))
PTAVE_BIN_SET = 'standard'      # test or standard
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.3, 2.4, 3.0)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
REBIN_INCLUSIVE_ETA = 2
REBIN_DIJET_ETA = 2
NORMALIZATION = 'integral'  # none, integral, or bin_width; never applied before F/B
INCLUSIVE_Y_RANGE = None
INCLUSIVE_FB_Y_RANGE = (0.75, 1.40)
DIJET_Y_RANGE = None
DIJET_OVERLAY_Y_RANGE = None
FB_Y_RANGE = (0.75, 1.25)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = Path(os.environ.get('MC_CP5_CHECK_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis' / 'output' / 'mc_cp5_check'))

CERNBOX_CP5_FILE = Path.home() / 'cernbox/ana/pPb8160/pythia/cp5tune/HiForestMiniAOD_hist.root'
LOCAL_CP5_FILE = Path.home() / 'cernbox/ana/pPb8160/pythia/cp5tune/cp5_gen_dijet_histograms.root'
EXPLICIT_CP5_FILE = os.environ.get('CP5_GEN_ANALYSIS_FILE')

if INPUT_SOURCE not in {'auto', 'cernbox', 'local'}:
    raise ValueError("INPUT_SOURCE must be 'auto', 'cernbox', or 'local'")
if PTAVE_BIN_SET not in {'test', 'standard'}:
    raise ValueError("PTAVE_BIN_SET must be 'test' or 'standard'")
if NORMALIZATION not in {'none', 'integral', 'bin_width'}:
    raise ValueError("NORMALIZATION must be 'none', 'integral', or 'bin_width'")
if ETA_CUT_INDEX not in range(len(ETA_CUTS)):
    raise ValueError(f'ETA_CUT_INDEX must be in [0, {len(ETA_CUTS)-1}]')
for label, intervals in {'INCLUSIVE_PT_BINS': INCLUSIVE_PT_BINS}.items():
    if not intervals or any(low >= high for low, high in intervals):
        raise ValueError(f'{label} must contain intervals satisfying low < high')
for label, value in {'REBIN_INCLUSIVE_ETA': REBIN_INCLUSIVE_ETA, 'REBIN_DIJET_ETA': REBIN_DIJET_ETA}.items():
    if isinstance(value, bool) or not isinstance(value, int) or value < 1:
        raise ValueError(f'{label} must be a positive integer')

if EXPLICIT_CP5_FILE:
    CP5_FILE = Path(EXPLICIT_CP5_FILE).expanduser()
elif INPUT_SOURCE == 'cernbox':
    CP5_FILE = CERNBOX_CP5_FILE
elif INPUT_SOURCE == 'local':
    CP5_FILE = LOCAL_CP5_FILE
else:
    candidates = (CERNBOX_CP5_FILE, LOCAL_CP5_FILE)
    CP5_FILE = next((path for path in candidates if path.exists()), candidates[0])
if not CP5_FILE.is_file():
    raise FileNotFoundError(f'Missing CP5 ROOT file: {CP5_FILE}')

PTAVE_BINS = tuple(TEST_DIJET_PTAVE_BINS if PTAVE_BIN_SET == 'test' else DIJET_PTAVE_BINS)
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
print('CP5 input:', CP5_FILE)
print(f'Per-jet eta cut: |etaCM| < {ETA_CUT:g} (index {ETA_CUT_INDEX})')
print(f'pTave bins ({PTAVE_BIN_SET}):', PTAVE_BINS)

## Inclusive generator jets

Project `hGenInclusiveJetPtEtaCM` onto $\eta_{CM}$ for every configured half-open jet-$p_T$ interval. Each full-distribution curve is normalized independently according to `NORMALIZATION`. The corresponding F/B curve is folded from the raw projection before normalization: positive $\eta_{CM}$ supplies Forward and reflected negative $\eta_{CM}$ supplies Backward. Their stored weighted uncertainties are divided with ROOT's independent-error option. The standalone input contains only jets above its generation-time storage threshold (normally 30 GeV), so intervals must not be interpreted below that threshold.

In [ ]:
# Cell role: project inclusive etaCM and construct F/B in every configured jet-pT range.
# Interpretation: shape normalization is display-only; F/B uses the unnormalized weighted projections.
inclusive_source = load_histogram(str(CP5_FILE), 'hGenInclusiveJetPtEtaCM')
if not inclusive_source.InheritsFrom('TH2'):
    raise TypeError('hGenInclusiveJetPtEtaCM must be a TH2')
inclusive_source.Rebin2D(1, REBIN_INCLUSIVE_ETA)
inclusive_shapes = {}
inclusive_fb_ratios = {}
for index, (low, high) in enumerate(INCLUSIVE_PT_BINS):
    projection = project_semantic_th2(inclusive_source, 'eta', (low, high), name=f'cp5_inclusive_eta_{index}')
    label = f'{low:g} < p_{{T}} < {high:g} GeV'
    inclusive_shapes[label] = normalize_histogram(projection, NORMALIZATION)
    inclusive_fb_ratios[label] = forward_backward_from_full(projection, name=f'cp5_inclusive_fb_{index}')

inclusive_normalization_titles = {'none': 'Weighted jets', 'integral': '1/N dN/d#eta_{CM}', 'bin_width': '1/N dN/d#eta_{CM}'}
dijet_normalization_titles = {'none': 'Weighted dijets', 'integral': '1/N dN/d#eta_{CM}^{dijet}', 'bin_width': '1/N dN/d#eta_{CM}^{dijet}'}
inclusive_canvas = draw_overlay(
    inclusive_shapes, title='', x_title='#eta_{CM}', y_title=inclusive_normalization_titles[NORMALIZATION],
    x_range=(-4.0, 4.0), y_range=INCLUSIVE_Y_RANGE, grid=DRAW_GRID,
    annotations=('pp 8.16 TeV', 'PYTHIA 8 CP5', 'anti-k_{T} R=0.4 gen jets'),
    legend_bounds=(0.60, 0.72, 0.88, 0.88), legend_text_size=0.020, headroom=1.8,
    output=OUTPUT_DIR / 'cp5_gen_inclusive_eta.pdf', save_png=SAVE_PNG,
    canvas_name='cp5_gen_inclusive_eta',
)
inclusive_fb_canvas = draw_overlay(
    inclusive_fb_ratios, title='', x_title='|#eta_{CM}|', y_title='Forward / Backward',
    x_range=(0.0, 4.0), y_range=INCLUSIVE_FB_Y_RANGE, grid=DRAW_GRID, reference_y=1.0,
    annotations=('pp 8.16 TeV', 'PYTHIA 8 CP5', 'anti-k_{T} R=0.4 gen jets'),
    legend_bounds=(0.60, 0.72, 0.88, 0.88), legend_text_size=0.020,
    output=OUTPUT_DIR / 'cp5_gen_inclusive_fb.pdf', save_png=SAVE_PNG,
    canvas_name='cp5_gen_inclusive_fb',
)
display(inclusive_canvas)
display(inclusive_fb_canvas)

## Generator dijets and Forward/Backward ratios

For each selected $p_T^{ave}$ interval, the shared closure helper projects `hGenDijetPtEtaCM_{index}` and its stored `Forward` and `Backward` companions over identical bins. The full distribution may be normalized for display. Forward and backward are kept unnormalized and divided with ROOT's independent-error option `''`; their stored `Sumw2` uncertainties are propagated.

In [ ]:
# Cell role: build the full dijet eta shape and unnormalized-yield F/B ratio for every pTave interval.
# Interpretation: F/B tests pp forward/backward symmetry; it is not an efficiency and never uses binomial errors.
GEN_CURVE = DijetClosureCurve(
    'Gen', 'hGenDijetPtEtaCM_{eta_cut_index}',
    'hGenDijetPtEtaForward_{eta_cut_index}',
    'hGenDijetPtEtaBackward_{eta_cut_index}',
)
DIJET_CACHE = {}
for ptave_range in PTAVE_BINS:
    eta_shapes, fb_ratios, selected_keys = build_dijet_gen_comparisons(
        CP5_FILE, (GEN_CURVE,), eta_cut_index=ETA_CUT_INDEX, ptave_range=ptave_range,
        nominal='Gen', rebin_eta=REBIN_DIJET_ETA, normalization=NORMALIZATION, ratio_option='',
    )
    DIJET_CACHE[tuple(ptave_range)] = {
        'eta': eta_shapes['Gen'], 'fb': fb_ratios['Gen'], 'keys': selected_keys['Gen'],
    }
print(f'Prepared {len(DIJET_CACHE)} dijet intervals with keys:', next(iter(DIJET_CACHE.values()))['keys'])

In [ ]:
# Cell role: overlay all dijet etaCM shapes, then draw one etaCM distribution and one F/B plot per pTave interval.
# Interpretation: indexed colors/markers distinguish pTave intervals; deviations of F/B from unity indicate forward/backward asymmetry.
dijet_overlay_shapes = {
    f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV': DIJET_CACHE[(low, high)]['eta']
    for low, high in PTAVE_BINS
}
dijet_overlay_styles = {label: index for index, label in enumerate(dijet_overlay_shapes)}
if PTAVE_BIN_SET == 'standard':
    dijet_overlay_legend_bounds = (0.38, 0.55, 0.88, 0.88)
    dijet_overlay_legend_columns = 2
    dijet_overlay_legend_text_size = 0.012
    dijet_overlay_headroom = 3.0
else:
    dijet_overlay_legend_bounds = (0.62, 0.62, 0.88, 0.88)
    dijet_overlay_legend_columns = 1
    dijet_overlay_legend_text_size = 0.018
    dijet_overlay_headroom = 2.3
dijet_overlay_canvas = draw_overlay(
    dijet_overlay_shapes, title='', x_title='#eta_{CM}^{dijet}',
    y_title=dijet_normalization_titles[NORMALIZATION], x_range=(-ETA_CUT, ETA_CUT),
    y_range=DIJET_OVERLAY_Y_RANGE, grid=DRAW_GRID,
    annotations=(
        'pp 8.16 TeV', 'PYTHIA 8 CP5', 'Gen dijets',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        'p_{T}^{Lead} #geq 50 GeV', 'p_{T}^{SubLead} #geq 40 GeV',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    ),
    style_indices=dijet_overlay_styles,
    legend_bounds=dijet_overlay_legend_bounds, legend_columns=dijet_overlay_legend_columns,
    legend_text_size=dijet_overlay_legend_text_size, headroom=dijet_overlay_headroom,
    output=OUTPUT_DIR / f'cp5_gen_dijet_eta_overlay_{PTAVE_BIN_SET}.pdf', save_png=SAVE_PNG,
    canvas_name=f'cp5_gen_dijet_eta_overlay_{PTAVE_BIN_SET}',
)
display(dijet_overlay_canvas)

dijet_results = {}
for low, high in PTAVE_BINS:
    entry = DIJET_CACHE[(low, high)]
    tag = f'{low:g}_{high:g}'.replace('.', 'p')
    annotations = (
        'pp 8.16 TeV', 'PYTHIA 8 CP5', 'Gen dijets',
        f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        'p_{T}^{Lead} #geq 50 GeV', 'p_{T}^{SubLead} #geq 40 GeV',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    eta_canvas = draw_overlay(
        {'Gen dijet': entry['eta']}, title='', x_title='#eta_{CM}^{dijet}',
        y_title=dijet_normalization_titles[NORMALIZATION], x_range=(-ETA_CUT, ETA_CUT),
        y_range=DIJET_Y_RANGE, grid=DRAW_GRID, annotations=annotations,
        show_legend=False,
        output=OUTPUT_DIR / f'cp5_gen_dijet_eta_ptave_{tag}.pdf', save_png=SAVE_PNG,
        canvas_name=f'cp5_gen_dijet_eta_ptave_{tag}',
    )
    fb_canvas = draw_overlay(
        {'Gen F/B': entry['fb']}, title='', x_title='|#eta_{CM}^{dijet}|',
        y_title='Forward / Backward', x_range=(0.0, ETA_CUT), y_range=FB_Y_RANGE,
        grid=DRAW_GRID, reference_y=1.0, annotations=annotations,
        show_legend=False,
        output=OUTPUT_DIR / f'cp5_gen_dijet_fb_ptave_{tag}.pdf', save_png=SAVE_PNG,
        canvas_name=f'cp5_gen_dijet_fb_ptave_{tag}', style_indices={'Gen F/B': 2},
    )
    dijet_results[(low, high)] = {'eta_canvas': eta_canvas, 'fb_canvas': fb_canvas, **entry}
    display(eta_canvas)
    display(fb_canvas)